# 📊 ML V2 - Comparação Gold V1 vs Gold V2

## 🎯 Objetivo

Medir **exclusivamente o impacto das 7 novas features** da Gold V2:

1. `rsi_14d`
2. `price_vs_ma_7d`
3. `price_vs_ma_30d`
4. `beta_90d` (corrigido)
5. `outperform_rate_30d`
6. `rolling_max_drawdown`
7. `dividend_stability`

## 📋 Comparação Controlada

**Fixo (não varia):**
* ✅ Mesmos 6.095 registros
* ✅ Mesmo `target_7d`
* ✅ Mesmo split temporal
* ✅ Mesmos hiperparâmetros XGBoost
* ✅ Mesma seed (42)
* ✅ Mesmo pré-processamento

**Variável (testado):**
* 🔬 Gold V1: 23 features base
* 🔬 Gold V2: 23 features base + 7 novas = 30 features

## 🎯 Baseline a Superar

**XGBoost + Gold V1 (notebook 33_ml_advanced):**
* ROC-AUC Test: **0.6366**
* Meta: ≥ 0.6466 (+0.01 ponto absoluto)

## ❓ Perguntas a Responder

1. A Gold V2 superou 0.6366 no Test?
2. O ganho foi ≥ 0.01 ponto absoluto?
3. A V2 melhorou também no Validation?
4. Alguma nova feature não agregou valor?
5. Manter V2, versão reduzida ou voltar para V1?
6. Qual versão seguir para walk-forward e tuning?

In [0]:
%pip install xgboost --quiet

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, 
    recall_score, f1_score, roc_curve, confusion_matrix, 
    ConfusionMatrixDisplay
)
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Bibliotecas importadas")
print(f"XGBoost version: {xgb.__version__}")

---

# 📦 1. CARREGAMENTO E VALIDAÇÃO DOS DADOS

## Validações Obrigatórias

✅ 6.095 registros em ambas as versões  
✅ Mesmas datas e tickers  
✅ Targets idênticos linha a linha  
✅ Mesma distribuição temporal

In [0]:
%sql
SELECT *
FROM workspace.gold.fii_features_v1
ORDER BY date, ticker

In [0]:
df_v1 = _sqldf.toPandas()

print("=" * 70)
print("GOLD V1 - workspace.gold.fii_features_v1")
print("=" * 70)
print(f"Registros: {len(df_v1):,}")
print(f"Período: {df_v1['date'].min()} até {df_v1['date'].max()}")
print(f"Tickers: {df_v1['ticker'].nunique()} ({sorted(df_v1['ticker'].unique())})")
print(f"Features: {len(df_v1.columns)}")
print(f"\nTarget target_7d:")
print(df_v1['target_7d'].value_counts())
print("=" * 70)

In [0]:
%sql
SELECT *
FROM workspace.gold.fii_features_v2
ORDER BY date, ticker

In [0]:
df_v2 = _sqldf.toPandas()

print("=" * 70)
print("GOLD V2 - workspace.gold.fii_features_v2")
print("=" * 70)
print(f"Registros: {len(df_v2):,}")
print(f"Período: {df_v2['date'].min()} até {df_v2['date'].max()}")
print(f"Tickers: {df_v2['ticker'].nunique()} ({sorted(df_v2['ticker'].unique())})")
print(f"Features: {len(df_v2.columns)}")
print(f"\nTarget target_7d:")
print(df_v2['target_7d'].value_counts())
print(f"\n7 Novas features:")
novas_features = ['rsi_14d', 'price_vs_ma_7d', 'price_vs_ma_30d', 'beta_90d', 
                   'outperform_rate_30d', 'rolling_max_drawdown', 'dividend_stability']
for feat in novas_features:
    if feat in df_v2.columns:
        nulls = df_v2[feat].isna().sum()
        pct_nulls = nulls / len(df_v2) * 100
        print(f"  {feat}: {nulls} nulos ({pct_nulls:.2f}%)")
print("=" * 70)

In [0]:
print("=" * 70)
print("VALIDAÇÃO: GOLD V1 vs GOLD V2")
print("=" * 70)

# 1. Mesma quantidade de registros
assert len(df_v1) == len(df_v2), f"ERRO: V1 tem {len(df_v1)} e V2 tem {len(df_v2)} registros"
print(f"✅ Mesma quantidade: {len(df_v1):,} registros")

# 2. Mesmas datas e tickers
df_v1_sorted = df_v1.sort_values(['date', 'ticker']).reset_index(drop=True)
df_v2_sorted = df_v2.sort_values(['date', 'ticker']).reset_index(drop=True)

assert (df_v1_sorted['date'] == df_v2_sorted['date']).all(), "ERRO: Datas diferentes"
assert (df_v1_sorted['ticker'] == df_v2_sorted['ticker']).all(), "ERRO: Tickers diferentes"
print(f"✅ Mesmas datas e tickers")

# 3. Targets idênticos
assert (df_v1_sorted['target_7d'] == df_v2_sorted['target_7d']).all(), "ERRO: Targets diferentes"
print(f"✅ Targets idênticos linha a linha")

# 4. Período
assert df_v1['date'].min() == df_v2['date'].min(), "ERRO: Data mínima diferente"
assert df_v1['date'].max() == df_v2['date'].max(), "ERRO: Data máxima diferente"
print(f"✅ Mesmo período: {df_v1['date'].min()} até {df_v1['date'].max()}")

print("\n" + "=" * 70)
print("✅✅✅ TODAS AS VALIDAÇÕES PASSARAM")
print("=" * 70)

---

# ✂️ 2. SPLIT TEMPORAL

## Split Obrigatório (mesmo do 33_ml_advanced)

* 🟢 **Train:** 2020-03-01 até 2022-12-31
* 🟡 **Validation:** 2023-01-01 até 2023-12-31
* 🔴 **Test:** 2024-01-01 até 2025-02-14 (out-of-sample)

⚠️ Test NUNCA deve ser usado para tuning ou imputação

In [0]:
# Converter datas
df_v1['date'] = pd.to_datetime(df_v1['date'])
df_v2['date'] = pd.to_datetime(df_v2['date'])

# Máscaras temporais
train_mask = (df_v1['date'] >= '2020-03-01') & (df_v1['date'] <= '2022-12-31')
val_mask = (df_v1['date'] >= '2023-01-01') & (df_v1['date'] <= '2023-12-31')
test_mask = (df_v1['date'] >= '2024-01-01') & (df_v1['date'] <= '2025-02-14')

print("=" * 70)
print("SPLIT TEMPORAL")
print("=" * 70)
print(f"\n🟢 TRAIN: 2020-03-01 até 2022-12-31")
print(f"   Registros: {train_mask.sum():,}")
print(f"   Target 1: {df_v1[train_mask]['target_7d'].sum()} ({df_v1[train_mask]['target_7d'].mean()*100:.1f}%)")

print(f"\n🟡 VALIDATION: 2023-01-01 até 2023-12-31")
print(f"   Registros: {val_mask.sum():,}")
print(f"   Target 1: {df_v1[val_mask]['target_7d'].sum()} ({df_v1[val_mask]['target_7d'].mean()*100:.1f}%)")

print(f"\n🔴 TEST: 2024-01-01 até 2025-02-14")
print(f"   Registros: {test_mask.sum():,}")
print(f"   Target 1: {df_v1[test_mask]['target_7d'].sum()} ({df_v1[test_mask]['target_7d'].mean()*100:.1f}%)")

# Validar ausência de sobreposição
total = train_mask.sum() + val_mask.sum() + test_mask.sum()
assert total == len(df_v1), f"ERRO: Sobreposição ou missing ({total} != {len(df_v1)})"
print(f"\n✅ Ausência de sobreposição: {total:,} = {len(df_v1):,}")
print("=" * 70)

---

# 🛠️ 3. PREPARAÇÃO DOS DADOS

## Gold V1: 23 features base

Remover: `ticker`, `date`, `target_alpha_7d`, `target_7d`

## Gold V2: 30 features (23 base + 7 novas)

Remover: `ticker`, `date`, `target_alpha_7d`, `target_7d`

## Tratamento de NaN em beta_90d

XGBoost suporta NaN nativamente - preservar como NaN.

In [0]:
# Colunas a remover
features_to_drop = ['ticker', 'date', 'target_alpha_7d']

# Gold V1
X_v1 = df_v1.drop(columns=features_to_drop + ['target_7d'])
y_v1 = df_v1['target_7d']

print("=" * 70)
print("GOLD V1 - PREPARAÇÃO")
print("=" * 70)
print(f"Features: {X_v1.shape[1]}")
print(f"Registros: {len(X_v1):,}")
print(f"\nFeatures V1:")
for i, col in enumerate(X_v1.columns, 1):
    nulls = X_v1[col].isna().sum()
    print(f"  {i:2d}. {col:25s} - {nulls:4d} nulos ({nulls/len(X_v1)*100:5.2f}%)")
print("=" * 70)

In [0]:
# Gold V2
X_v2 = df_v2.drop(columns=features_to_drop + ['target_7d'])
y_v2 = df_v2['target_7d']

# Converter outperform_rate_30d para float (estava como object)
if 'outperform_rate_30d' in X_v2.columns:
    X_v2['outperform_rate_30d'] = pd.to_numeric(X_v2['outperform_rate_30d'], errors='coerce')

print("=" * 70)
print("GOLD V2 - PREPARAÇÃO")
print("=" * 70)
print(f"Features: {X_v2.shape[1]}")
print(f"Registros: {len(X_v2):,}")
print(f"\nFeatures V2:")
for i, col in enumerate(X_v2.columns, 1):
    nulls = X_v2[col].isna().sum()
    novo = "  🆕" if col in ['rsi_14d', 'price_vs_ma_7d', 'price_vs_ma_30d', 'beta_90d', 
                                 'outperform_rate_30d', 'rolling_max_drawdown', 'dividend_stability'] else ""
    print(f"  {i:2d}. {col:25s} - {nulls:4d} nulos ({nulls/len(X_v2)*100:5.2f}%){novo}")
print("=" * 70)

In [0]:
# Split V1
X_v1_train, y_v1_train = X_v1[train_mask], y_v1[train_mask]
X_v1_val, y_v1_val = X_v1[val_mask], y_v1[val_mask]
X_v1_test, y_v1_test = X_v1[test_mask], y_v1[test_mask]

print("=" * 70)
print("GOLD V1 - SPLIT")
print("=" * 70)
print(f"Train: X={X_v1_train.shape}, y={y_v1_train.shape}")
print(f"Val:   X={X_v1_val.shape}, y={y_v1_val.shape}")
print(f"Test:  X={X_v1_test.shape}, y={y_v1_test.shape}")
print("=" * 70)

In [0]:
# Split V2
X_v2_train, y_v2_train = X_v2[train_mask], y_v2[train_mask]
X_v2_val, y_v2_val = X_v2[val_mask], y_v2[val_mask]
X_v2_test, y_v2_test = X_v2[test_mask], y_v2[test_mask]

print("=" * 70)
print("GOLD V2 - SPLIT")
print("=" * 70)
print(f"Train: X={X_v2_train.shape}, y={y_v2_train.shape}")
print(f"Val:   X={X_v2_val.shape}, y={y_v2_val.shape}")
print(f"Test:  X={X_v2_test.shape}, y={y_v2_test.shape}")
print("=" * 70)

In [0]:
print("=" * 70)
print("VALIDAÇÃO: TARGETS V1 vs V2")
print("=" * 70)

assert (y_v1_train == y_v2_train).all(), "ERRO: Targets Train diferentes"
assert (y_v1_val == y_v2_val).all(), "ERRO: Targets Val diferentes"
assert (y_v1_test == y_v2_test).all(), "ERRO: Targets Test diferentes"

print("✅ Targets Train idênticos (V1 == V2)")
print("✅ Targets Val idênticos (V1 == V2)")
print("✅ Targets Test idênticos (V1 == V2)")
print("=" * 70)

---

# 🚀 4. MODELO V1 (GOLD V1 + XGBOOST)

## Hiperparâmetros (do 33_ml_advanced)

```python
n_estimators=200
max_depth=6
learning_rate=0.05
subsample=0.8
colsample_bytree=0.8
random_state=42
eval_metric='auc'
early_stopping_rounds=20
```

## Baseline a Superar

ROC-AUC Test: **0.6366**

In [0]:
print("=" * 70)
print("🚀 TREINANDO XGBOOST V1 (GOLD V1)")
print("=" * 70)

# Hiperparâmetros exatos do 33_ml_advanced
xgb_v1 = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc',
    early_stopping_rounds=20,
    verbosity=0
)

xgb_v1.fit(
    X_v1_train, y_v1_train,
    eval_set=[(X_v1_val, y_v1_val)],
    verbose=False
)

print("✅ XGBoost V1 treinado")
print(f"Best iteration: {xgb_v1.best_iteration}")
print(f"Best score (Val AUC): {xgb_v1.best_score:.4f}")
print("=" * 70)

In [0]:
# Predições V1
y_v1_val_proba = xgb_v1.predict_proba(X_v1_val)[:, 1]
y_v1_val_pred = xgb_v1.predict(X_v1_val)

y_v1_test_proba = xgb_v1.predict_proba(X_v1_test)[:, 1]
y_v1_test_pred = xgb_v1.predict(X_v1_test)

print("✅ Predições V1 geradas")

In [0]:
# Métricas V1
v1_metrics = {
    'val_auc': roc_auc_score(y_v1_val, y_v1_val_proba),
    'val_acc': accuracy_score(y_v1_val, y_v1_val_pred),
    'val_prec': precision_score(y_v1_val, y_v1_val_pred),
    'val_rec': recall_score(y_v1_val, y_v1_val_pred),
    'val_f1': f1_score(y_v1_val, y_v1_val_pred),
    'test_auc': roc_auc_score(y_v1_test, y_v1_test_proba),
    'test_acc': accuracy_score(y_v1_test, y_v1_test_pred),
    'test_prec': precision_score(y_v1_test, y_v1_test_pred),
    'test_rec': recall_score(y_v1_test, y_v1_test_pred),
    'test_f1': f1_score(y_v1_test, y_v1_test_pred)
}

print("=" * 70)
print("📊 XGBOOST V1 - RESULTADOS")
print("=" * 70)
print(f"\n🟡 VALIDATION")
print(f"ROC-AUC:   {v1_metrics['val_auc']:.4f}")
print(f"Accuracy:  {v1_metrics['val_acc']:.4f}")
print(f"Precision: {v1_metrics['val_prec']:.4f}")
print(f"Recall:    {v1_metrics['val_rec']:.4f}")
print(f"F1-Score:  {v1_metrics['val_f1']:.4f}")

print(f"\n🔴 TEST (OUT-OF-SAMPLE)")
print(f"ROC-AUC:   {v1_metrics['test_auc']:.4f}  ← Baseline: 0.6366")
print(f"Accuracy:  {v1_metrics['test_acc']:.4f}")
print(f"Precision: {v1_metrics['test_prec']:.4f}")
print(f"Recall:    {v1_metrics['test_rec']:.4f}")
print(f"F1-Score:  {v1_metrics['test_f1']:.4f}")
print("=" * 70)

---

# 🚀 5. MODELO V2 (GOLD V2 + XGBOOST)

## 7 Novas Features

1. `rsi_14d`
2. `price_vs_ma_7d`
3. `price_vs_ma_30d`
4. `beta_90d` (corrigido - 1.90% nulos preservados)
5. `outperform_rate_30d`
6. `rolling_max_drawdown`
7. `dividend_stability`

## Mesmos Hiperparâmetros

Idênticos ao V1 (seed=42)

In [0]:
print("=" * 70)
print("🚀 TREINANDO XGBOOST V2 (GOLD V2)")
print("=" * 70)

# Mesmos hiperparâmetros, mesma seed
xgb_v2 = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc',
    early_stopping_rounds=20,
    verbosity=0
)

xgb_v2.fit(
    X_v2_train, y_v2_train,
    eval_set=[(X_v2_val, y_v2_val)],
    verbose=False
)

print("✅ XGBoost V2 treinado")
print(f"Best iteration: {xgb_v2.best_iteration}")
print(f"Best score (Val AUC): {xgb_v2.best_score:.4f}")
print("=" * 70)

In [0]:
# Predições V2
y_v2_val_proba = xgb_v2.predict_proba(X_v2_val)[:, 1]
y_v2_val_pred = xgb_v2.predict(X_v2_val)

y_v2_test_proba = xgb_v2.predict_proba(X_v2_test)[:, 1]
y_v2_test_pred = xgb_v2.predict(X_v2_test)

print("✅ Predições V2 geradas")

In [0]:
# Métricas V2
v2_metrics = {
    'val_auc': roc_auc_score(y_v2_val, y_v2_val_proba),
    'val_acc': accuracy_score(y_v2_val, y_v2_val_pred),
    'val_prec': precision_score(y_v2_val, y_v2_val_pred),
    'val_rec': recall_score(y_v2_val, y_v2_val_pred),
    'val_f1': f1_score(y_v2_val, y_v2_val_pred),
    'test_auc': roc_auc_score(y_v2_test, y_v2_test_proba),
    'test_acc': accuracy_score(y_v2_test, y_v2_test_pred),
    'test_prec': precision_score(y_v2_test, y_v2_test_pred),
    'test_rec': recall_score(y_v2_test, y_v2_test_pred),
    'test_f1': f1_score(y_v2_test, y_v2_test_pred)
}

print("=" * 70)
print("📊 XGBOOST V2 - RESULTADOS")
print("=" * 70)
print(f"\n🟡 VALIDATION")
print(f"ROC-AUC:   {v2_metrics['val_auc']:.4f}")
print(f"Accuracy:  {v2_metrics['val_acc']:.4f}")
print(f"Precision: {v2_metrics['val_prec']:.4f}")
print(f"Recall:    {v2_metrics['val_rec']:.4f}")
print(f"F1-Score:  {v2_metrics['val_f1']:.4f}")

print(f"\n🔴 TEST (OUT-OF-SAMPLE)")
print(f"ROC-AUC:   {v2_metrics['test_auc']:.4f}  ← Meta: 0.6466 (+0.01)")
print(f"Accuracy:  {v2_metrics['test_acc']:.4f}")
print(f"Precision: {v2_metrics['test_prec']:.4f}")
print(f"Recall:    {v2_metrics['test_rec']:.4f}")
print(f"F1-Score:  {v2_metrics['test_f1']:.4f}")
print("=" * 70)

---

# 🔬 6. COMPARAÇÃO V1 vs V2

## Objetivo

Medir o **impacto exclusivo** das 7 novas features:
* Diferença absoluta (V2 - V1)
* Diferença relativa ((V2 - V1) / V1 * 100%)

In [0]:
# Criar DataFrame comparativo
comparison = pd.DataFrame({
    'Metric': ['ROC-AUC', 'Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'V1_Val': [
        v1_metrics['val_auc'], v1_metrics['val_acc'], 
        v1_metrics['val_prec'], v1_metrics['val_rec'], v1_metrics['val_f1']
    ],
    'V2_Val': [
        v2_metrics['val_auc'], v2_metrics['val_acc'], 
        v2_metrics['val_prec'], v2_metrics['val_rec'], v2_metrics['val_f1']
    ],
    'V1_Test': [
        v1_metrics['test_auc'], v1_metrics['test_acc'], 
        v1_metrics['test_prec'], v1_metrics['test_rec'], v1_metrics['test_f1']
    ],
    'V2_Test': [
        v2_metrics['test_auc'], v2_metrics['test_acc'], 
        v2_metrics['test_prec'], v2_metrics['test_rec'], v2_metrics['test_f1']
    ]
})

# Calcular diferenças
comparison['Diff_Val'] = comparison['V2_Val'] - comparison['V1_Val']
comparison['Diff_Test'] = comparison['V2_Test'] - comparison['V1_Test']
comparison['Rel_Val%'] = (comparison['Diff_Val'] / comparison['V1_Val'] * 100).round(2)
comparison['Rel_Test%'] = (comparison['Diff_Test'] / comparison['V1_Test'] * 100).round(2)

print("=" * 100)
print("🔬 COMPARAÇÃO V1 vs V2")
print("=" * 100)
print(comparison.to_string(index=False))
print("=" * 100)

In [0]:
print("=" * 70)
print("🎯 RESUMO EXECUTIVO")
print("=" * 70)

# ROC-AUC Test
auc_diff = v2_metrics['test_auc'] - v1_metrics['test_auc']
auc_rel = (auc_diff / v1_metrics['test_auc']) * 100
baseline = 0.6366
meta = 0.6466

print(f"\n🎯 ROC-AUC TEST (métrica principal)")
print(f"   V1 (Gold V1):     {v1_metrics['test_auc']:.4f}")
print(f"   V2 (Gold V2):     {v2_metrics['test_auc']:.4f}")
print(f"   Diferença:        {auc_diff:+.4f} ({auc_rel:+.2f}%)")
print(f"\n   Baseline 33_ml:   {baseline:.4f}")
print(f"   Meta (+0.01):     {meta:.4f}")

if v2_metrics['test_auc'] > baseline:
    print(f"\n   ✅ V2 SUPEROU baseline ({v2_metrics['test_auc']:.4f} > {baseline:.4f})")
else:
    print(f"\n   ❌ V2 NÃO superou baseline ({v2_metrics['test_auc']:.4f} ≤ {baseline:.4f})")

if auc_diff >= 0.01:
    print(f"   ✅ Ganho ≥ 0.01 ponto absoluto")
else:
    print(f"   ❌ Ganho < 0.01 ponto absoluto")

# ROC-AUC Validation
auc_val_diff = v2_metrics['val_auc'] - v1_metrics['val_auc']
print(f"\n🟡 ROC-AUC VALIDATION")
print(f"   Diferença: {auc_val_diff:+.4f} ({(auc_val_diff/v1_metrics['val_auc']*100):+.2f}%)")

if auc_val_diff > 0 and auc_diff > 0:
    print("   ✅ V2 melhorou em ambos (Val e Test)")
elif auc_diff > 0:
    print("   ⚠️ V2 melhorou apenas no Test (Val piorou)")
else:
    print("   ❌ V2 não melhorou")

print("=" * 70)

In [0]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Curva ROC - Validation
fpr_v1_val, tpr_v1_val, _ = roc_curve(y_v1_val, y_v1_val_proba)
fpr_v2_val, tpr_v2_val, _ = roc_curve(y_v2_val, y_v2_val_proba)

axes[0, 0].plot(fpr_v1_val, tpr_v1_val, label=f'V1 (AUC={v1_metrics["val_auc"]:.4f})', linewidth=2)
axes[0, 0].plot(fpr_v2_val, tpr_v2_val, label=f'V2 (AUC={v2_metrics["val_auc"]:.4f})', linewidth=2)
axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[0, 0].set_xlabel('False Positive Rate', fontsize=12)
axes[0, 0].set_ylabel('True Positive Rate', fontsize=12)
axes[0, 0].set_title('🟡 ROC Curve - Validation', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(alpha=0.3)

# 2. Curva ROC - Test
fpr_v1_test, tpr_v1_test, _ = roc_curve(y_v1_test, y_v1_test_proba)
fpr_v2_test, tpr_v2_test, _ = roc_curve(y_v2_test, y_v2_test_proba)

axes[0, 1].plot(fpr_v1_test, tpr_v1_test, label=f'V1 (AUC={v1_metrics["test_auc"]:.4f})', linewidth=2)
axes[0, 1].plot(fpr_v2_test, tpr_v2_test, label=f'V2 (AUC={v2_metrics["test_auc"]:.4f})', linewidth=2)
axes[0, 1].axhline(y=0.6366, color='red', linestyle='--', alpha=0.5, label='Baseline 33_ml (0.6366)')
axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[0, 1].set_xlabel('False Positive Rate', fontsize=12)
axes[0, 1].set_ylabel('True Positive Rate', fontsize=12)
axes[0, 1].set_title('🔴 ROC Curve - Test (Out-of-Sample)', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(alpha=0.3)

# 3. Matriz de Confusão V1 Test
cm_v1 = confusion_matrix(y_v1_test, y_v1_test_pred)
disp_v1 = ConfusionMatrixDisplay(confusion_matrix=cm_v1, display_labels=['Negativo', 'Positivo'])
disp_v1.plot(ax=axes[1, 0], cmap='Blues', colorbar=False)
axes[1, 0].set_title('V1 - Confusion Matrix (Test)', fontsize=14, fontweight='bold')
axes[1, 0].grid(False)

# 4. Matriz de Confusão V2 Test
cm_v2 = confusion_matrix(y_v2_test, y_v2_test_pred)
disp_v2 = ConfusionMatrixDisplay(confusion_matrix=cm_v2, display_labels=['Negativo', 'Positivo'])
disp_v2.plot(ax=axes[1, 1], cmap='Greens', colorbar=False)
axes[1, 1].set_title('V2 - Confusion Matrix (Test)', fontsize=14, fontweight='bold')
axes[1, 1].grid(False)

plt.tight_layout()
plt.show()

print("✅ Visualizações geradas")

   
---

# 📊 7. ANÁLISE DE FEATURE IMPORTANCE

## Objetivo

Rankear as 30 features da Gold V2 por importância no XGBoost e identificar:
* Quais das 7 novas features agregaram valor
* Quais features antigas continuam dominantes
* Se alguma feature nova ficou irrelevante

---

# 🔬 8. ABLATION STUDY - SELEÇÃO DE FEATURES

## 🎯 Objetivo

Identificar quais das 7 novas features da Gold V2 realmente agregam valor, usando **exclusivamente Train e Validation**.

## 📋 Metodologia

**Baseline:**
* Gold V1 (23 features base)
* XGBoost com mesmos hiperparâmetros do 33_ml_advanced
* Métricas: ROC-AUC Validation

**Experimentos:**
1. **Individual:** V1 + cada feature nova isoladamente (7 testes)
2. **Combinações:** V1 + features que melhoraram individualmente
3. **Subconjunto final:** features com ganho consistente

## ⚠️ Regras Estritas

* ✅ Usar apenas Train e Validation para decisões
* ❌ Test NÃO será usado para seleção
* ✅ Mesmo XGBoost, mesma seed (42)
* ✅ Mesmo split temporal

## 📊 Deliverables

* ROC-AUC Validation de cada experimento
* Diferença em relação à V1 baseline
* Ranking por ganho incremental
* Subconjunto mínimo recomendado
* Features a descartar

In [0]:
def train_and_evaluate(X_train, y_train, X_val, y_val, feature_set_name):
    """
    Treina XGBoost com hiperparâmetros fixos e retorna ROC-AUC Validation.
    
    Args:
        X_train: Features de treino
        y_train: Target de treino
        X_val: Features de validação
        y_val: Target de validação
        feature_set_name: Nome descritivo do conjunto de features
    
    Returns:
        dict com métricas e modelo
    """
    # Mesmos hiperparâmetros do 33_ml_advanced
    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='auc',
        early_stopping_rounds=20,
        verbosity=0
    )
    
    # Treinar
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    # Predições Validation
    y_val_proba = model.predict_proba(X_val)[:, 1]
    y_val_pred = model.predict(X_val)
    
    # Métricas Validation
    val_auc = roc_auc_score(y_val, y_val_proba)
    val_acc = accuracy_score(y_val, y_val_pred)
    val_prec = precision_score(y_val, y_val_pred)
    val_rec = recall_score(y_val, y_val_pred)
    val_f1 = f1_score(y_val, y_val_pred)
    
    return {
        'feature_set': feature_set_name,
        'n_features': X_train.shape[1],
        'val_auc': val_auc,
        'val_acc': val_acc,
        'val_prec': val_prec,
        'val_rec': val_rec,
        'val_f1': val_f1,
        'best_iteration': model.best_iteration,
        'model': model
    }

print("✅ Função auxiliar criada")

In [0]:
# Lista para armazenar resultados
ablation_results = []

print("=" * 70)
print("📊 EXPERIMENTO 1: BASELINE - GOLD V1 (23 FEATURES)")
print("=" * 70)

# Treinar baseline V1
baseline_result = train_and_evaluate(
    X_v1_train, y_v1_train,
    X_v1_val, y_v1_val,
    "Baseline V1 (23 features)"
)

ablation_results.append(baseline_result)

print(f"\nFeatures: {baseline_result['n_features']}")
print(f"ROC-AUC Val: {baseline_result['val_auc']:.4f}")
print(f"Accuracy Val: {baseline_result['val_acc']:.4f}")
print(f"Best iteration: {baseline_result['best_iteration']}")
print("\n✅ Baseline V1 treinado")
print("=" * 70)

In [0]:
# 7 features candidatas
novas_features_candidatas = [
    'rsi_14d',
    'price_vs_ma_7d', 
    'price_vs_ma_30d',
    'beta_90d',
    'outperform_rate_30d',
    'rolling_max_drawdown',
    'dividend_stability'
]

print("=" * 70)
print("🔬 EXPERIMENTOS 2-8: TESTES INDIVIDUAIS (V1 + 1 FEATURE)")
print("=" * 70)

for i, new_feature in enumerate(novas_features_candidatas, start=2):
    print(f"\n🧪 Experimento {i}: V1 + {new_feature}")
    
    # Adicionar a nova feature à V1 (usar X_v2 já preprocessado)
    X_train_with_feature = pd.concat([
        X_v1_train.reset_index(drop=True),
        X_v2_train[[new_feature]].reset_index(drop=True)
    ], axis=1)
    
    X_val_with_feature = pd.concat([
        X_v1_val.reset_index(drop=True),
        X_v2_val[[new_feature]].reset_index(drop=True)
    ], axis=1)
    
    # Treinar e avaliar
    result = train_and_evaluate(
        X_train_with_feature, y_v1_train,
        X_val_with_feature, y_v1_val,
        f"V1 + {new_feature}"
    )
    
    ablation_results.append(result)
    
    # Calcular diferença em relação ao baseline
    diff = result['val_auc'] - baseline_result['val_auc']
    diff_pct = (diff / baseline_result['val_auc']) * 100
    
    print(f"  ROC-AUC Val: {result['val_auc']:.4f} (Diff: {diff:+.4f} / {diff_pct:+.2f}%)")
    
print("\n" + "=" * 70)
print("✅ Testes individuais concluídos")
print("=" * 70)

In [0]:
# Criar DataFrame com resultados dos testes individuais
individual_results = pd.DataFrame(ablation_results)

# Calcular diferenças em relação ao baseline
baseline_auc = individual_results.iloc[0]['val_auc']
individual_results['diff_auc'] = individual_results['val_auc'] - baseline_auc
individual_results['diff_auc_pct'] = (individual_results['diff_auc'] / baseline_auc) * 100

print("=" * 90)
print("📊 RESULTADOS DOS TESTES INDIVIDUAIS")
print("=" * 90)
print(individual_results[['feature_set', 'n_features', 'val_auc', 'diff_auc', 'diff_auc_pct']].to_string(index=False))

# Identificar features que melhoraram (diff_auc > 0)
features_que_melhoraram = []

print("\n" + "=" * 90)
print("✅ FEATURES QUE MELHORARAM O VALIDATION (diff_auc > 0)")
print("=" * 90)

for idx, row in individual_results[1:].iterrows():  # Pular o baseline
    if row['diff_auc'] > 0:
        feature_name = row['feature_set'].replace('V1 + ', '')
        features_que_melhoraram.append(feature_name)
        print(f"  ✅ {feature_name:30s}  AUC: {row['val_auc']:.4f}  Diff: {row['diff_auc']:+.4f} ({row['diff_auc_pct']:+.2f}%)")

if not features_que_melhoraram:
    print("  ❌ Nenhuma feature melhorou individualmente")

# Features que pioraram ou não mudaram
print("\n" + "=" * 90)
print("❌ FEATURES QUE PIORARAM OU NÃO MUDARAM (diff_auc <= 0)")
print("=" * 90)

for idx, row in individual_results[1:].iterrows():
    if row['diff_auc'] <= 0:
        feature_name = row['feature_set'].replace('V1 + ', '')
        print(f"  ❌ {feature_name:30s}  AUC: {row['val_auc']:.4f}  Diff: {row['diff_auc']:+.4f} ({row['diff_auc_pct']:+.2f}%)")

print("\n" + "=" * 90)
print(f"\nResumo: {len(features_que_melhoraram)} features melhoraram, {7 - len(features_que_melhoraram)} não melhoraram")
print("=" * 90)

In [0]:
print("=" * 70)
print("🧩 EXPERIMENTOS DE COMBINAÇÕES")
print("=" * 70)

if len(features_que_melhoraram) == 0:
    print("\n⚠️ Nenhuma feature melhorou individualmente.")
    print("Não há combinações para testar.")
    
elif len(features_que_melhoraram) == 1:
    print(f"\n✅ Apenas 1 feature melhorou: {features_que_melhoraram[0]}")
    print("Não há necessidade de testar combinações.")
    
else:
    # Testar todas as features que melhoraram juntas
    print(f"\n🧪 Experimento: V1 + TODAS as features que melhoraram ({len(features_que_melhoraram)})")
    print(f"Features: {', '.join(features_que_melhoraram)}")
    
    # Criar dataset com todas as features que melhoraram (usar X_v2 já preprocessado)
    X_train_combined = pd.concat([
        X_v1_train.reset_index(drop=True),
        X_v2_train[features_que_melhoraram].reset_index(drop=True)
    ], axis=1)
    
    X_val_combined = pd.concat([
        X_v1_val.reset_index(drop=True),
        X_v2_val[features_que_melhoraram].reset_index(drop=True)
    ], axis=1)
    
    # Treinar e avaliar
    combined_result = train_and_evaluate(
        X_train_combined, y_v1_train,
        X_val_combined, y_v1_val,
        f"V1 + {len(features_que_melhoraram)} features"
    )
    
    ablation_results.append(combined_result)
    
    diff = combined_result['val_auc'] - baseline_auc
    diff_pct = (diff / baseline_auc) * 100
    
    print(f"\n  ROC-AUC Val: {combined_result['val_auc']:.4f}")
    print(f"  Diff vs Baseline: {diff:+.4f} ({diff_pct:+.2f}%)")
    
    # Comparar com os melhores individuais
    best_individual = individual_results[individual_results['feature_set'].str.contains('V1 \\+')]['val_auc'].max()
    diff_vs_best = combined_result['val_auc'] - best_individual
    
    print(f"  Melhor individual: {best_individual:.4f}")
    print(f"  Diff vs Melhor individual: {diff_vs_best:+.4f}")
    
    if diff_vs_best > 0:
        print("  ✅ Combinação SUPEROU o melhor individual (sinergia positiva)")
    elif diff_vs_best < 0:
        print("  ⚠️ Combinação PIOROU em relação ao melhor individual (sinergia negativa)")
    else:
        print("  ↔️ Combinação igual ao melhor individual")

print("\n" + "=" * 70)
print("✅ Testes de combinações concluídos")
print("=" * 70)

In [0]:
# Atualizar DataFrame com todos os resultados
final_results = pd.DataFrame(ablation_results)
final_results['diff_auc'] = final_results['val_auc'] - baseline_auc
final_results['diff_auc_pct'] = (final_results['diff_auc'] / baseline_auc) * 100

# Ordenar por ROC-AUC Validation (decrescente)
final_results_sorted = final_results.sort_values('val_auc', ascending=False).reset_index(drop=True)
final_results_sorted['rank'] = range(1, len(final_results_sorted) + 1)

print("=" * 100)
print("🏆 RANKING FINAL - TODOS OS EXPERIMENTOS")
print("=" * 100)
print(final_results_sorted[['rank', 'feature_set', 'n_features', 'val_auc', 'diff_auc', 'diff_auc_pct']].to_string(index=False))
print("=" * 100)

# Identificar o melhor modelo
best_model = final_results_sorted.iloc[0]

print("\n" + "=" * 100)
print("⭐ MELHOR MODELO")
print("=" * 100)
print(f"Feature Set: {best_model['feature_set']}")
print(f"Número de features: {best_model['n_features']}")
print(f"ROC-AUC Val: {best_model['val_auc']:.4f}")
print(f"Ganho vs Baseline: {best_model['diff_auc']:+.4f} ({best_model['diff_auc_pct']:+.2f}%)")
print("=" * 100)

# Análise de estabilidade: comparar val_acc, val_f1
print("\n" + "=" * 100)
print("📊 ANÁLISE DE ESTABILIDADE (Top 5)")
print("=" * 100)
top5 = final_results_sorted.head(5)
print(top5[['rank', 'feature_set', 'val_auc', 'val_acc', 'val_f1']].to_string(index=False))
print("\nNotas:")
print("  - Modelos estáveis: métricas consistentes (AUC, Accuracy, F1)")
print("  - Modelos instáveis: AUC alto mas Accuracy/F1 baixos (overfitting ou threshold ruim)")
print("=" * 100)

In [0]:
print("=" * 100)
print("📝 RECOMENDAÇÕES FINAIS - ABLATION STUDY")
print("=" * 100)

# 1. Baseline
print(f"\n📊 Baseline V1 (23 features)")
print(f"   ROC-AUC Val: {baseline_auc:.4f}")

# 2. Features que agregam valor
print(f"\n✅ Features que AGREGAM VALOR (melhoram Validation individualmente):")
if len(features_que_melhoraram) > 0:
    for feature in features_que_melhoraram:
        feature_result = individual_results[individual_results['feature_set'] == f'V1 + {feature}'].iloc[0]
        print(f"   • {feature:30s}  Ganho: {feature_result['diff_auc']:+.4f} ({feature_result['diff_auc_pct']:+.2f}%)")
else:
    print("   ❌ Nenhuma feature melhorou individualmente")

# 3. Features a descartar
print(f"\n❌ Features a DESCARTAR (pioram ou não mudam Validation):")
features_a_descartar = []
for idx, row in individual_results[1:].iterrows():
    if row['diff_auc'] <= 0:
        feature_name = row['feature_set'].replace('V1 + ', '')
        features_a_descartar.append(feature_name)
        print(f"   • {feature_name:30s}  Ganho: {row['diff_auc']:+.4f} ({row['diff_auc_pct']:+.2f}%)")

if len(features_a_descartar) == 0:
    print("   ✅ Todas as features agregam valor")

# 4. Subconjunto mínimo recomendado
print(f"\n🎯 SUBCONJUNTO MÍNIMO RECOMENDADO:")

if best_model['feature_set'] == 'Baseline V1 (23 features)':
    print("   ⚠️ Recomendação: MANTER GOLD V1 (23 features)")
    print("   Motivo: Nenhuma feature nova melhorou o Validation")
    print("   Ação: Descartar todas as 7 novas features")
    
elif len(features_que_melhoraram) == 1:
    print(f"   ✅ Gold V1 + {features_que_melhoraram[0]}")
    print(f"   Total: 24 features (23 base + 1 nova)")
    print(f"   ROC-AUC Val esperado: {individual_results[individual_results['feature_set'] == f'V1 + {features_que_melhoraram[0]}'].iloc[0]['val_auc']:.4f}")
    
elif len(features_que_melhoraram) > 1:
    # Verificar se combinação foi testada
    combined_model = final_results_sorted[final_results_sorted['feature_set'].str.contains('features') & 
                                           (final_results_sorted['n_features'] > 24)]
    
    if len(combined_model) > 0:
        combined_model = combined_model.iloc[0]
        
        # Comparar combinação vs melhor individual
        best_individual_in_list = individual_results[individual_results['feature_set'].str.contains('V1 \\+')]['val_auc'].max()
        
        if combined_model['val_auc'] > best_individual_in_list:
            print(f"   ✅ Gold V1 + {len(features_que_melhoraram)} features novas (COMBINAÇÃO)")
            print(f"   Features: {', '.join(features_que_melhoraram)}")
            print(f"   Total: {combined_model['n_features']} features")
            print(f"   ROC-AUC Val: {combined_model['val_auc']:.4f}")
            print(f"   Sinergia: POSITIVA (combinação > melhor individual)")
        else:
            best_individual_feature = individual_results[individual_results['val_auc'] == best_individual_in_list].iloc[0]['feature_set'].replace('V1 + ', '')
            print(f"   ⚠️ Gold V1 + {best_individual_feature} (APENAS MELHOR INDIVIDUAL)")
            print(f"   Total: 24 features")
            print(f"   ROC-AUC Val: {best_individual_in_list:.4f}")
            print(f"   Motivo: Combinação não trouxe ganho adicional (sinergia neutra/negativa)")
    else:
        # Recomendar melhor individual
        best_individual_in_list = individual_results[individual_results['feature_set'].str.contains('V1 \\+')]['val_auc'].max()
        best_individual_feature = individual_results[individual_results['val_auc'] == best_individual_in_list].iloc[0]['feature_set'].replace('V1 + ', '')
        print(f"   ✅ Gold V1 + {best_individual_feature}")
        print(f"   Total: 24 features")
        print(f"   ROC-AUC Val: {best_individual_in_list:.4f}")

# 5. Próximos passos
print(f"\n🚀 PRÓXIMOS PASSOS:")
print(f"   1. Implementar o subconjunto recomendado acima")
print(f"   2. Testar no Test set (APENAS UMA VEZ, no final)")
print(f"   3. Se ROC-AUC Test > 0.6366, seguir para walk-forward e tuning")
print(f"   4. Se ROC-AUC Test ≤ 0.6366, reavaliar feature engineering")

print("\n" + "=" * 100)
print("✅ ABLATION STUDY CONCLUÍDA")
print("=" * 100)

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 1. Gráfico de barras: ROC-AUC Validation de todos os experimentos
final_results_plot = final_results_sorted.copy()
final_results_plot['feature_set_short'] = final_results_plot['feature_set'].str.replace('V1 + ', '').str[:25]

colors = ['red' if row['feature_set'] == 'Baseline V1 (23 features)' else 
          'green' if row['val_auc'] == final_results_sorted['val_auc'].max() else 
          'lightblue' for _, row in final_results_plot.iterrows()]

axes[0].barh(final_results_plot['feature_set_short'], final_results_plot['val_auc'], color=colors, alpha=0.7)
axes[0].axvline(x=baseline_auc, color='red', linestyle='--', linewidth=2, label=f'Baseline ({baseline_auc:.4f})')
axes[0].set_xlabel('ROC-AUC Validation', fontsize=12)
axes[0].set_title('📊 ROC-AUC Validation - Ablation Study', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(axis='x', alpha=0.3)
axes[0].invert_yaxis()

# 2. Gráfico de ganho incremental (apenas testes individuais)
individual_only = final_results_sorted[final_results_sorted['feature_set'].str.contains('V1 \\+') & 
                                        (final_results_sorted['n_features'] == 24)].copy()

if len(individual_only) > 0:
    individual_only['feature_name'] = individual_only['feature_set'].str.replace('V1 + ', '')
    individual_only_sorted = individual_only.sort_values('diff_auc', ascending=True)
    
    colors_gain = ['green' if x > 0 else 'red' for x in individual_only_sorted['diff_auc']]
    
    axes[1].barh(individual_only_sorted['feature_name'], individual_only_sorted['diff_auc'], color=colors_gain, alpha=0.7)
    axes[1].axvline(x=0, color='black', linestyle='-', linewidth=1)
    axes[1].set_xlabel('Ganho ROC-AUC vs Baseline', fontsize=12)
    axes[1].set_title('🔬 Ganho Incremental - Testes Individuais', fontsize=14, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)
    axes[1].invert_yaxis()
else:
    axes[1].text(0.5, 0.5, 'Nenhum teste individual realizado', 
                ha='center', va='center', fontsize=14, transform=axes[1].transAxes)
    axes[1].axis('off')

plt.tight_layout()
plt.show()

print("✅ Visualizações geradas")

---

# 🏁 RESUMO EXECUTIVO - ABLATION STUDY

## 📊 Resultados Principais

### Baseline (Gold V1)
* **23 features**
* **ROC-AUC Validation: 0.6030**

### Melhor Modelo
* **⭐ Gold V1 + price_vs_ma_7d**
* **24 features (23 base + 1 nova)**
* **ROC-AUC Validation: 0.6156**
* **Ganho: +0.0126 (+2.08%)**

---

## ✅ Features que AGREGAM VALOR

| Feature | Ganho AUC | Ganho % | Ranking |
|---------|-----------|---------|----------|
| **price_vs_ma_7d** | **+0.0126** | **+2.08%** | **#1** |
| rolling_max_drawdown | +0.0086 | +1.43% | #2 |

---

## ❌ Features a DESCARTAR (5 features)

| Feature | Ganho AUC | Ganho % | Motivo |
|---------|-----------|---------|--------|
| **price_vs_ma_30d** | **-0.0168** | **-2.79%** | **Pior performance** |
| dividend_stability | -0.0146 | -2.42% | Piora modelo |
| beta_90d | -0.0142 | -2.35% | Piora modelo |
| outperform_rate_30d | -0.0125 | -2.07% | Piora modelo |
| rsi_14d | -0.0059 | -0.98% | Piora modelo |

---

## ⚠️ Sinergia Negativa

**Teste de combinação:**
* V1 + price_vs_ma_7d + rolling_max_drawdown
* ROC-AUC Val: **0.5976** (PIOR que baseline!)
* Diferença vs melhor individual: **-0.0179**

**Conclusão:** As 2 features que melhoraram individualmente **interferem negativamente** quando combinadas. A solução ótima é usar **apenas price_vs_ma_7d**.

---

## 🎯 RECOMENDAÇÃO FINAL

### Subconjunto Mínimo:
```
Gold V1 + price_vs_ma_7d
```

**Total:** 24 features (23 base + 1 nova)

**ROC-AUC Validation esperado:** 0.6156

**Descarte:**
* rsi_14d
* price_vs_ma_30d
* beta_90d
* outperform_rate_30d
* rolling_max_drawdown *(apesar de melhorar individualmente, criará sinergia negativa)*
* dividend_stability

---

## 🚀 PRÓXIMOS PASSOS

1. **Criar Gold V2_minimal** com as 24 features recomendadas
2. **Treinar modelo final** com Gold V2_minimal
3. **Avaliar no Test set** (APENAS UMA VEZ)
4. **Meta:** ROC-AUC Test > 0.6366 (baseline do notebook 33_ml_advanced)
5. Se meta atingida: **seguir para walk-forward e tuning**
6. Se meta não atingida: **reavaliar feature engineering**

---

## 📝 Observações Importantes

* ✅ Test NÃO foi usado para seleção (apenas Train + Validation)
* ✅ Todos os experimentos usaram mesmos hiperparâmetros (seed=42)
* ✅ Split temporal preservado
* 🚨 **5 das 7 features novas pioraram o modelo**
* 🚨 **Combinação de features boas pode criar sinergia negativa**
* ✅ **price_vs_ma_7d** foi a única feature que trouxe ganho sólido e estável

In [0]:
# Tabela comparativa final com estabilidade
print("=" * 110)
print("📋 TABELA COMPARATIVA FINAL - ABLATION STUDY (9 EXPERIMENTOS)")
print("=" * 110)

final_comparison = final_results_sorted[[
    'rank', 'feature_set', 'n_features', 'val_auc', 'val_acc', 
    'val_prec', 'val_rec', 'val_f1', 'diff_auc', 'diff_auc_pct'
]].copy()

print(final_comparison.to_string(index=False))

print("\n" + "=" * 110)
print("LEGENDA:")
print("  Rank 1-2: Features que MELHORAM o modelo (ganho positivo)")
print("  Rank 3:   BASELINE V1 (referência)")
print("  Rank 4-9: Features que PIORAM o modelo (ganho negativo)")
print("=" * 110)

# Estatísticas finais
print("\n" + "=" * 110)
print("📊 ESTATÍSTICAS DA ABLATION STUDY")
print("=" * 110)
print(f"Total de experimentos: {len(final_results_sorted)}")
print(f"  - 1 Baseline (V1)")
print(f"  - 7 Testes individuais (V1 + 1 feature)")
print(f"  - 1 Combinação (V1 + 2 features que melhoraram)")
print(f"\nResultados:")
print(f"  ✅ Features que melhoraram: 2 (price_vs_ma_7d, rolling_max_drawdown)")
print(f"  ❌ Features que pioraram: 5")
print(f"  ⚠️ Combinação: sinergia NEGATIVA")
print(f"\nMelhor modelo:")
print(f"  ⭐ Gold V1 + price_vs_ma_7d")
print(f"  📊 ROC-AUC Val: 0.6156 (+2.08% vs baseline)")
print(f"  🎯 Features: 24 (23 base + 1 nova)")
print("\n" + "=" * 110)
print("✅ ABLATION STUDY FINALIZADA COM SUCESSO")
print("=" * 110)

In [0]:
# Extrair feature importance do modelo V2
importances = xgb_v2.feature_importances_
feature_names = X_v2.columns

# Criar DataFrame
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

# Marcar novas features
novas_features = ['rsi_14d', 'price_vs_ma_7d', 'price_vs_ma_30d', 'beta_90d', 
                   'outperform_rate_30d', 'rolling_max_drawdown', 'dividend_stability']
feature_importance_df['Nova'] = feature_importance_df['Feature'].isin(novas_features)
feature_importance_df['Rank'] = range(1, len(feature_importance_df) + 1)

print("=" * 80)
print("📊 FEATURE IMPORTANCE - GOLD V2 (30 FEATURES)")
print("=" * 80)
print("\nTop 30 (todas as features):")
for idx, row in feature_importance_df.iterrows():
    novo_flag = "  🆕" if row['Nova'] else ""
    print(f"  {row['Rank']:2d}. {row['Feature']:30s}  Importance: {row['Importance']:.6f}{novo_flag}")

print("\n" + "=" * 80)
print("🆕 RANKING DAS 7 NOVAS FEATURES")
print("=" * 80)
novas_df = feature_importance_df[feature_importance_df['Nova']].copy()
for idx, row in novas_df.iterrows():
    print(f"  Rank {row['Rank']:2d}: {row['Feature']:30s}  Importance: {row['Importance']:.6f}")

print("\n" + "=" * 80)
print("📈 ANÁLISE")
print("=" * 80)

# Estatísticas das novas features
novas_rank_medio = novas_df['Rank'].mean()
novas_importance_total = novas_df['Importance'].sum()
total_importance = feature_importance_df['Importance'].sum()
novas_pct = (novas_importance_total / total_importance) * 100

print(f"\nNovas features (7 features, 23.3% do total):")
print(f"  Rank médio: {novas_rank_medio:.1f}")
print(f"  Importância total: {novas_importance_total:.6f} ({novas_pct:.2f}% do total)")
print(f"  Melhor posição: {novas_df['Rank'].min()} ({novas_df.iloc[0]['Feature']})")
print(f"  Pior posição: {novas_df['Rank'].max()} ({novas_df.iloc[-1]['Feature']})")

# Identificar features irrelevantes (importance < threshold)
threshold = 0.01
irrelevantes = feature_importance_df[feature_importance_df['Importance'] < threshold]
novas_irrelevantes = irrelevantes[irrelevantes['Nova']]

if len(novas_irrelevantes) > 0:
    print(f"\n⚠️ Novas features com importance < {threshold}:")
    for idx, row in novas_irrelevantes.iterrows():
        print(f"  - {row['Feature']}: {row['Importance']:.6f} (Rank {row['Rank']})")
else:
    print(f"\n✅ Todas as 7 novas features têm importance ≥ {threshold}")

print("=" * 80)

In [0]:
# Gráfico de feature importance (top 20)
fig, ax = plt.subplots(figsize=(12, 10))

top_n = 20
top_features = feature_importance_df.head(top_n).copy()
top_features = top_features.sort_values('Importance', ascending=True)  # Inverter para gráfico horizontal

colors = ['#2ecc71' if nova else '#3498db' for nova in top_features['Nova']]

ax.barh(range(len(top_features)), top_features['Importance'], color=colors)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'], fontsize=10)
ax.set_xlabel('Importance', fontsize=12, fontweight='bold')
ax.set_title(f'📊 Feature Importance - Top {top_n} Features\n🆕 Verde = Novas | 🔵 Azul = Base', 
             fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"✅ Visualização das top {top_n} features")

   
---

# ✅ 8. CONCLUSÕES E RECOMENDAÇÕES

## Resposta às Perguntas de Negócio

Baseadas nos resultados empíricos acima.

In [0]:
print("=" * 80)
print("✅ RESPOSTAS ÀS PERGUNTAS DE NEGÓCIO")
print("=" * 80)

# 1. A Gold V2 superou 0.6366 no Test?
baseline = 0.6366
v2_test_auc = v2_metrics['test_auc']
superou = v2_test_auc > baseline

print(f"\n1️⃣ A Gold V2 superou 0.6366 no Test?")
if superou:
    print(f"   ✅ SIM: {v2_test_auc:.4f} > {baseline:.4f} (ganho: {v2_test_auc - baseline:+.4f})")
else:
    print(f"   ❌ NÃO: {v2_test_auc:.4f} ≤ {baseline:.4f} (diferença: {v2_test_auc - baseline:+.4f})")

# 2. O ganho foi ≥ 0.01 ponto absoluto?
auc_diff = v2_test_auc - v1_metrics['test_auc']
ganho_significativo = auc_diff >= 0.01

print(f"\n2️⃣ O ganho foi ≥ 0.01 ponto absoluto?")
if ganho_significativo:
    print(f"   ✅ SIM: Ganho de {auc_diff:+.4f} pontos ({(auc_diff/v1_metrics['test_auc']*100):+.2f}%)")
else:
    print(f"   ❌ NÃO: Ganho de apenas {auc_diff:+.4f} pontos ({(auc_diff/v1_metrics['test_auc']*100):+.2f}%)")

# 3. A V2 melhorou também no Validation?
auc_val_diff = v2_metrics['val_auc'] - v1_metrics['val_auc']
melhorou_val = auc_val_diff > 0

print(f"\n3️⃣ A V2 melhorou também no Validation?")
if melhorou_val:
    print(f"   ✅ SIM: Ganho de {auc_val_diff:+.4f} pontos ({(auc_val_diff/v1_metrics['val_auc']*100):+.2f}%)")
else:
    print(f"   ❌ NÃO: Perda de {auc_val_diff:+.4f} pontos ({(auc_val_diff/v1_metrics['val_auc']*100):+.2f}%)")

# 4. Alguma nova feature não agregou valor?
print(f"\n4️⃣ Alguma nova feature não agregou valor?")
threshold = 0.01
novas_features_list = ['rsi_14d', 'price_vs_ma_7d', 'price_vs_ma_30d', 'beta_90d', 
                        'outperform_rate_30d', 'rolling_max_drawdown', 'dividend_stability']
novas_imp = feature_importance_df[feature_importance_df['Feature'].isin(novas_features_list)]
irrelevantes = novas_imp[novas_imp['Importance'] < threshold]

if len(irrelevantes) > 0:
    print(f"   ⚠️ SIM: {len(irrelevantes)} feature(s) com importance < {threshold}:")
    for idx, row in irrelevantes.iterrows():
        print(f"      - {row['Feature']}: {row['Importance']:.6f} (Rank {row['Rank']})")
else:
    print(f"   ✅ NÃO: Todas as 7 novas features têm importance ≥ {threshold}")
    print(f"   Rank médio das novas: {novas_imp['Rank'].mean():.1f}")

print("=" * 80)

In [0]:
print("=" * 80)
print("🎯 RECOMENDAÇÃO FINAL")
print("=" * 80)

# Decisão baseada nos resultados
v2_test_auc = v2_metrics['test_auc']
v1_test_auc = v1_metrics['test_auc']
auc_diff = v2_test_auc - v1_test_auc
baseline = 0.6366

# Critérios de decisão
superou_baseline = v2_test_auc > baseline
ganho_significativo = auc_diff >= 0.01
melhorou_val = (v2_metrics['val_auc'] - v1_metrics['val_auc']) > 0

print("\n📊 Critérios avaliados:")
print(f"  {'  ✅' if superou_baseline else '  ❌'} V2 superou baseline 33_ml (0.6366)")
print(f"  {'  ✅' if ganho_significativo else '  ❌'} Ganho ≥ 0.01 ponto absoluto vs V1")
print(f"  {'  ✅' if melhorou_val else '  ❌'} Melhorou também no Validation")

# Decisão
if superou_baseline and ganho_significativo:
    decisao = "✅ ADOTAR GOLD V2"
    justificativa = f"""A Gold V2 cumpriu os critérios:
  - ROC-AUC Test: {v2_test_auc:.4f} (vs baseline {baseline:.4f})
  - Ganho absoluto: {auc_diff:+.4f} pontos vs V1
  - As 7 novas features agregaram valor mensurável
  
  🚀 Próximos passos:
  - Usar Gold V2 como base para walk-forward validation
  - Prosseguir com tuning de hiperparâmetros no V2
  - Considerar feature selection se alguma feature tiver importance muito baixa"""
elif auc_diff > 0:
    decisao = "⚠️ CONSIDERAR GOLD V2 (ganho marginal)"
    justificativa = f"""A Gold V2 melhorou, mas o ganho foi modesto:
  - ROC-AUC Test: {v2_test_auc:.4f} (vs V1 {v1_test_auc:.4f})
  - Ganho absoluto: {auc_diff:+.4f} pontos (< 0.01 ponto)
  
  🔍 Opções:
  - Testar Gold V2 com hiperparâmetros ajustados
  - Investigar feature engineering adicional
  - Considerar versão híbrida (V1 + subset de V2)"""
else:
    decisao = "❌ MANTER GOLD V1"
    justificativa = f"""A Gold V2 não melhorou:
  - ROC-AUC Test V1: {v1_test_auc:.4f}
  - ROC-AUC Test V2: {v2_test_auc:.4f}
  - Diferença: {auc_diff:+.4f} pontos
  
  🔍 Próximos passos:
  - Manter Gold V1 como base
  - Revisar cálculo das 7 novas features
  - Investigar outras feature engineering approaches"""

print(f"\n🎯 DECISÃO: {decisao}")
print(f"\n{justificativa}")
print("\n" + "=" * 80)
print("✅ ANÁLISE COMPLETA - NOTEBOOK 35_ml_v2 FINALIZADO")
print("=" * 80)

In [0]:
# Visualizar top 20 features
top_n = 20
top_features = importance_df.head(top_n).copy()

# Cores: verde para novas, azul para base
colors = ['#2ecc71' if feat in novas_features else '#3498db' for feat in top_features['Feature']]

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(range(len(top_features)), top_features['Importance'], color=colors)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance', fontsize=12)
ax.set_title(f'📊 Top {top_n} Features - Modelo V2', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Legenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='Novas (Gold V2)'),
    Patch(facecolor='#3498db', label='Base (Gold V1)')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11)

plt.tight_layout()
plt.show()

print("✅ Visualização de feature importance gerada")

---

# 🎯 8. CONCLUSÕES E RECOMENDAÇÕES

## Respondendo as 6 Perguntas-Chave

In [0]:
print("=" * 80)
print("🎯 CONCLUSÕES FINAIS - GOLD V1 vs GOLD V2")
print("=" * 80)

# Calcular métricas
auc_diff = v2_metrics['test_auc'] - v1_metrics['test_auc']
baseline_33ml = 0.6366
meta = 0.6466

print(f"\n❓ PERGUNTA 1: A Gold V2 superou o ROC-AUC Test de 0.6366?")
if v2_metrics['test_auc'] > baseline_33ml:
    print(f"   ✅ SIM! V2 = {v2_metrics['test_auc']:.4f} > {baseline_33ml:.4f} (baseline)")
    print(f"   Ganho: +{(v2_metrics['test_auc'] - baseline_33ml):.4f} pontos")
else:
    print(f"   ❌ NÃO. V2 = {v2_metrics['test_auc']:.4f} ≤ {baseline_33ml:.4f} (baseline)")
    print(f"   Perda: {(v2_metrics['test_auc'] - baseline_33ml):.4f} pontos")

print(f"\n❓ PERGUNTA 2: O ganho foi de pelo menos 0.01 ponto absoluto?")
if auc_diff >= 0.01:
    print(f"   ✅ SIM! Ganho V2-V1 = {auc_diff:+.4f} ≥ 0.01")
else:
    print(f"   ❌ NÃO. Ganho V2-V1 = {auc_diff:+.4f} < 0.01")

print(f"\n❓ PERGUNTA 3: A V2 melhorou também no Validation, ou somente no Test?")
auc_val_diff = v2_metrics['val_auc'] - v1_metrics['val_auc']
if auc_val_diff > 0 and auc_diff > 0:
    print(f"   ✅ Melhorou em AMBOS")
    print(f"      Validation: {auc_val_diff:+.4f} ({v1_metrics['val_auc']:.4f} → {v2_metrics['val_auc']:.4f})")
    print(f"      Test:       {auc_diff:+.4f} ({v1_metrics['test_auc']:.4f} → {v2_metrics['test_auc']:.4f})")
elif auc_diff > 0:
    print(f"   ⚠️ Melhorou APENAS no Test")
    print(f"      Validation: {auc_val_diff:+.4f} (PIOROU)")
    print(f"      Test:       {auc_diff:+.4f} (melhorou)")
else:
    print(f"   ❌ NÃO melhorou")
    print(f"      Validation: {auc_val_diff:+.4f}")
    print(f"      Test:       {auc_diff:+.4f}")

print(f"\n❓ PERGUNTA 4: Alguma nova feature não agregou valor?")
novas_importance = importance_df[importance_df['Feature'].isin(novas_features)].copy()
novas_importance['Rank'] = novas_importance.index + 1
baixas = novas_importance[novas_importance['Rank'] > 20]
if len(baixas) > 0:
    print(f"   ⚠️ SIM - {len(baixas)} feature(s) com baixa importância (rank > 20):")
    for idx, row in baixas.iterrows():
        print(f"      - {row['Feature']:25s}: rank {row['Rank']}/30, importance {row['Importance']:.4f}")
    print(f"\n   🔍 Candidatas à remoção futura (se sem ganho preditivo)")
else:
    print(f"   ✅ Todas as 7 novas features estão no top 20")

print("=" * 80)

In [0]:
print("=" * 80)
print("💡 RECOMENDAÇÃO FINAL")
print("=" * 80)

print(f"\n❓ PERGUNTA 5: Devemos manter V2, testar uma versão reduzida ou voltar para V1?")

# Decisão baseada nos resultados
if v2_metrics['test_auc'] > baseline_33ml and auc_diff >= 0.01:
    decisao = "MANTER V2"
    justificativa = f"V2 superou baseline ({v2_metrics['test_auc']:.4f} > {baseline_33ml:.4f}) com ganho ≥ 0.01"
elif v2_metrics['test_auc'] > v1_metrics['test_auc'] and v2_metrics['test_auc'] > baseline_33ml:
    decisao = "MANTER V2 (com ressalvas)"
    justificativa = f"V2 superou baseline mas ganho < 0.01 ({auc_diff:+.4f})"
elif len(baixas) > 0 and auc_diff > 0:
    decisao = "TESTAR VERSÃO REDUZIDA"
    justificativa = f"V2 melhorou mas {len(baixas)} feature(s) com baixa importância"
else:
    decisao = "VOLTAR PARA V1"
    justificativa = f"V2 não trouxe ganho preditivo consistente"

print(f"\n   🛡️ DECISÃO: {decisao}")
print(f"   📝 Justificativa: {justificativa}")

print(f"\n❓ PERGUNTA 6: Qual versão deve seguir para walk-forward e tuning?")
if decisao.startswith("MANTER V2"):
    print(f"   ➡️ Seguir com GOLD V2 para walk-forward e tuning")
    print(f"   🔧 Próximos passos:")
    print(f"      1. Walk-forward validation com V2")
    print(f"      2. Tuning de hiperparâmetros com V2")
    print(f"      3. Monitorar {len(novas_features)} novas features em produção")
elif decisao == "TESTAR VERSÃO REDUZIDA":
    print(f"   ➡️ Criar GOLD V2.1 (removendo features de baixa importância)")
    print(f"   🔧 Próximos passos:")
    print(f"      1. Criar Gold V2.1 sem: {', '.join(baixas['Feature'].tolist())}")
    print(f"      2. Re-testar Gold V2.1 vs V1 vs V2")
    print(f"      3. Seguir com a melhor versão para walk-forward")
else:
    print(f"   ➡️ Seguir com GOLD V1 para walk-forward e tuning")
    print(f"   🔧 Próximos passos:")
    print(f"      1. Walk-forward validation com V1")
    print(f"      2. Tuning de hiperparâmetros com V1")
    print(f"      3. Revisar Gold V2 - features não agregaram valor")

print(f"\n⚠️ NÃO avançar automaticamente para tuning - aguardar aprovação")
print("=" * 80)

---

# ✅ EXPERIMENTO CONCLUÍDO

## 📊 Resultados Principais

| Versão | ROC-AUC Val | ROC-AUC Test | Features |
|--------|-------------|--------------|----------|
| V1     | ?           | ?            | 23       |
| V2     | ?           | ?            | 30 (+7)  |

## 🔬 Método Científico Aplicado

✅ Controle rigoroso (mesmos registros, targets, split, hiperparâmetros)  
✅ Validações automáticas (6.095 registros, targets idênticos)  
✅ Métricas múltiplas (AUC, Accuracy, Precision, Recall, F1)  
✅ Feature importance analítica (7 novas features avaliadas)  
✅ Conclusões data-driven (baseadas em evidências empíricas)

## 🚀 Próxima Fase

Aguardar execução do notebook para resultados finais.